# Оценка качества сгенерированных упражнений: LLM-судья (Kimi K2.5)

Судья — `moonshotai/kimi-k2.5` через OpenRouter. Проходит по
`all_predictions_300_v2.json` (ответы 6 моделей-генераторов), по каждому из
300 упражнений применяет рубрику и возвращает по-критериальные оценки + вердикт.

Модели: `qwen2.5-3b`, `smollm3-3b`, `phi3.5-mini`,
`baseline_fib_v2`, `baseline_recon_v2`, `claude-opus-4-7`.

## 1. Установка и клиент OpenRouter

In [1]:
!pip install -q openai tqdm


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import os
import json
import time
from pathlib import Path

from openai import OpenAI
from tqdm.auto import tqdm

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY", "<OPENROUTER_API_KEY>"),
)

JUDGE_MODEL = "moonshotai/kimi-k2.5"

MAX_ITEMS = None     # None = все 300; число = быстрый смок-прогон
N_RETRIES = 4
CACHE_DIR = Path("judge_cache")
CACHE_DIR.mkdir(exist_ok=True)

/Users/rom4ik/Desktop/askarka/exgen/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Загрузка данных

`all_predictions_300_v2.json` — словарь
`{ "<model>": [ <pred_0>, ... <pred_299> ] }`. Каждый `pred` — это
JSON-**строка** контракта (econ-escaped), поэтому нужен повторный
`json.loads`. `val_meta.json` — список из 300 мет, выровненный по индексам;
используем из него ТОЛЬКО `source`.

In [2]:
PRED_PATH = "data/predictions/all_predictions_300_v3.json"
META_PATH = "data/dataset/val_meta.json"

with open(PRED_PATH, "r", encoding="utf-8") as f:
    PREDICTIONS = json.load(f)

with open(META_PATH, "r", encoding="utf-8") as f:
    VAL_META = json.load(f)

MODELS = ["qwen2.5-3b", "smollm3-3b", "phi3.5-mini",
          "baseline_fib_v2", "baseline_recon_v2", "claude-opus-4-7"]

for m in MODELS:
    if m not in PREDICTIONS:
        print(f"[warn] '{m}' отсутствует в predictions")
    else:
        n = len(PREDICTIONS[m])
        flag = "" if n == len(VAL_META) else f"  [!] != мета ({len(VAL_META)})"
        print(f"{m:<20} предсказаний: {n}{flag}")

qwen2.5-3b           предсказаний: 300
smollm3-3b           предсказаний: 300
phi3.5-mini          предсказаний: 300
baseline_fib_v2      предсказаний: 300
baseline_recon_v2    предсказаний: 300
claude-opus-4-7      предсказаний: 300


In [3]:
def to_exercise_text(pred):
    """pred -> строка контракта для судьи.

    В файле каждый pred — econ-escaped JSON-строка, поэтому пытаемся
    распарсить и нормализовать. Покрытие не считаем (300/300 у всех)."""
    if isinstance(pred, (dict, list)):
        return json.dumps(pred, ensure_ascii=False)
    s = str(pred).strip()
    try:
        obj = json.loads(s)                 # строка -> объект
        if isinstance(obj, str):            # двойная упаковка
            obj = json.loads(obj)
        return json.dumps(obj, ensure_ascii=False)
    except (json.JSONDecodeError, TypeError):
        return s                            # отдаём как есть; C7 поймает кривость

def get_source(idx):
    """Реальное предложение ученика с ошибкой — единственное, что берём
    из меты. Контекст реактивной персонализации для критерия C5."""
    if idx < len(VAL_META):
        return VAL_META[idx].get("source", "")
    return ""

## 3. Промпт судьи (чистый intrinsic; meta-поля судье не передаются)

In [4]:
JUDGE_SYSTEM = """You are an expert evaluator of English-language learning exercises.
You act ONLY as a judge. You do not rewrite, repair, complete, or improve the exercise.
You assess the exercise exactly as given, against the rubric below, and nothing else.
If the exercise is ambiguous or under-specified, that counts against it - do not resolve it charitably.

You will receive:
1. The learner's ORIGINAL sentence containing a real mistake (this is the
   context the exercise was generated to remediate).
2. One generated exercise as a JSON object with this contract:
   target_error_category, corrected_sentence,
   task.type, task.instruction_en,
   task.content_en.{context_text, items[*].{question_en, options_en, student_answer_en}, word_bank}.

WHAT THIS ARTIFACT IS - read before scoring:
This JSON is the TEACHER'S ANSWER KEY, not the student-facing rendering.
By the project's authoring spec:
- `**_word_**` is the intentional notation for the keyed correct answer.
  It is NEVER a defect on its own. Do not penalize its presence anywhere.
- For task.type = "vocabulary_fill", `context_text` may legitimately appear
  with the answers inlined as `**_answer_**`. This is by spec, NOT malformed.
- For "grammar_choice" and "transformation", `student_answer_en` is the FULL
  sentence with the keyed token in `**_..._**` (grammar_choice also shows
  the options in parentheses). That is the spec.
- The student-facing stem is obtained by replacing every `**_X_**` with "___".
  Judge stem clarity (C2) against THIS derived student view.
- `options_en = null` is correct for vocabulary_fill, transformation,
  reading_tf, writing_sample, matching. Only "functional_match" and
  "grammar_choice" carry per-item options.

You are NOT told which error category or task type was requested. Judge the
exercise on its own merits and on whether it remediates the mistake actually
present in the learner's original sentence.

Score each criterion independently on a 3-point scale:
0 = fails, 1 = borderline, 2 = fully meets. Give a one-sentence justification
for every score. Judge one criterion at a time; do not let one influence another.

C1. GRAMMATICAL CORRECTNESS - stems and keyed correct answers are grammatical
    English (ignoring the `**_..._**` markup itself).
C2. STEM CLARITY / NON-AMBIGUITY - the student-facing stem (with `**_X_**`
    replaced by "___") gives enough unambiguous context; the gap admits a
    single reading.
C3. SINGLE DEFENSIBLE KEY - exactly one answer is correct in context; no
    alternative is also acceptable. Empty `student_answer_en` where the task
    requires one, or a key that is wrong in context, fails.
C4. DISTRACTOR / OPTION QUALITY - apply per `task.type`:
      - grammar_choice: the inline options in parentheses inside
        `student_answer_en` must be plausible, grammatically parallel, not
        nonsense, not duplicates.
      - vocabulary_fill: judge the `word_bank` as the distractor pool -
        plausible, unique (no duplicates), category-consistent with the gaps.
        Do NOT penalize `options_en = null` here.
      - functional_match: each item must carry `options_en` with 4 labelled
        choices; judge those.
      - transformation, reading_tf, writing_sample, matching: not applicable
        by spec - score 2.
C5. RELEVANCE TO THE LEARNER'S ACTUAL MISTAKE - the exercise genuinely trains
    the specific kind of mistake present in the learner's ORIGINAL sentence,
    not some unrelated point.
C6. PEDAGOGICAL VALUE - useful for learning the targeted point at reasonable
    difficulty; has genuine instructional depth.
C7. STRUCTURAL WELL-FORMEDNESS - the object is internally coherent. Count as
    defects ONLY:
      - declared `task.type` does not match content shape (e.g. functional_match
        with no `options_en`);
      - item count violates the spec (vocabulary_fill: 12-20 gaps;
        grammar_choice: 10-14; transformation: 8-12; reading_tf: 8;
        functional_match: 8-10);
      - duplicate `question_en` strings that require DIFFERENT answers;
      - missing `student_answer_en` where the task type requires it;
      - `corrected_sentence` missing or not actually correcting the learner's
        error.
    Do NOT count as defects: presence of `**_..._**` markers anywhere;
    `context_text` with inlined answers for vocabulary_fill; `options_en=null`
    for task types that do not require options.

After scoring, output an OVERALL VERDICT, exactly one of:
    ACCEPT_AS_IS            - a teacher could use this item unchanged
    ACCEPT_WITH_MINOR_EDITS - usable after a small wording fix
    REJECT                  - not usable; a substantive defect
If REJECT, assign one PRIMARY_FAILURE tag from this closed list:
    grammatical | ambiguous_stem | multiple_correct | weak_distractor |
    off_target | low_pedagogy | malformed_structure

Do not compare this item to any other item. Do not average the criteria into a
single number yourself.

Return ONLY this JSON, nothing else:
{
  "scores": { "C1":0, "C2":0, "C3":0, "C4":0, "C5":0, "C6":0, "C7":0 },
  "justifications": { "C1":"...", "C2":"...", "C3":"...", "C4":"...",
                      "C5":"...", "C6":"...", "C7":"..." },
  "overall_verdict": "ACCEPT_AS_IS | ACCEPT_WITH_MINOR_EDITS | REJECT",
  "primary_failure": "none | grammatical | ambiguous_stem | multiple_correct | weak_distractor | off_target | low_pedagogy | malformed_structure"
}"""

JUDGE_USER_TEMPLATE = """Learner's original sentence (contains a real mistake):
{source}

Exercise to evaluate (verbatim):
{exercise}

Evaluate strictly per the rubric. Output only the required JSON."""

## 4. Вызов судьи: парсинг, ретраи, кэш

In [5]:
CRITERIA = ["C1", "C2", "C3", "C4", "C5", "C6", "C7"]
VERDICTS = {"ACCEPT_AS_IS", "ACCEPT_WITH_MINOR_EDITS", "REJECT"}

def _extract_json(text):
    if not text:
        raise ValueError("empty response")
    t = text.strip()
    if t.startswith("```"):
        t = t.split("```")[1]
        if t.lstrip().lower().startswith("json"):
            t = t.lstrip()[4:]
    a, b = t.find("{"), t.rfind("}")
    if a == -1 or b == -1 or b < a:
        raise ValueError("no JSON object found")
    return json.loads(t[a:b + 1])

def _valid(v):
    if not set(CRITERIA).issubset(set(v.get("scores", {}).keys())):
        return False
    return v.get("overall_verdict") in VERDICTS

def judge_one(exercise_text, source):
    user = JUDGE_USER_TEMPLATE.format(source=source or "(not provided)",
                                      exercise=exercise_text)
    for attempt in range(N_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "system", "content": JUDGE_SYSTEM},
                          {"role": "user", "content": user}],
                temperature=0,
                extra_body={"reasoning": {"enabled": False}},
            )
            v = _extract_json(resp.choices[0].message.content)
            if _valid(v):
                v["scores"] = {c: int(v["scores"][c]) for c in CRITERIA}
                return v
            raise ValueError("verdict failed schema check")
        except Exception as e:
            if attempt == N_RETRIES - 1:
                print(f"  [judge fail] {type(e).__name__}: {str(e)[:120]}")
                return None
            time.sleep(2 ** attempt)
    return None

def _cache_path(model):
    return CACHE_DIR / f"{model}.jsonl"

def _load_cache(model):
    p, cache = _cache_path(model), {}
    if p.exists():
        with open(p, "r", encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                cache[rec["idx"]] = rec["verdict"]
    return cache

def _append_cache(model, idx, verdict):
    with open(_cache_path(model), "a", encoding="utf-8") as f:
        f.write(json.dumps({"idx": idx, "verdict": verdict},
                           ensure_ascii=False) + "\n")

## 5. Метрики одной модели

Все 300 судятся (покрытие 100% у всех — coverage не считаем). `judge_fail` —
сколько упражнений судья не смог отсудить из-за сбоя API/парсинга.

In [6]:
VERDICT_VALUE = {"ACCEPT_AS_IS": 1.0, "ACCEPT_WITH_MINOR_EDITS": 0.5, "REJECT": 0.0}

def evaluate_model(name):
    preds = PREDICTIONS.get(name, [])
    n = len(preds) if MAX_ITEMS is None else min(MAX_ITEMS, len(preds))
    cache = _load_cache(name)

    judged, fail = [], 0
    for i in tqdm(range(n), desc=name):
        if i in cache:
            v = cache[i]
        else:
            v = judge_one(to_exercise_text(preds[i]), get_source(i))
            _append_cache(name, i, v)
        if v is None:
            fail += 1
            continue
        judged.append(v)

    nj = len(judged)
    out = {"model": name, "total": n, "judged_ok": nj, "judge_fail": fail}
    if nj == 0:
        for c in CRITERIA:
            out[c] = None
        out.update({"criteria_mean_0_1": None, "acceptance_score": None,
                    "pct_accept_as_is": None, "pct_usable": None,
                    "pct_reject": None, "top_failure": None})
        return out

    for c in CRITERIA:
        out[c] = round(sum(v["scores"][c] for v in judged) / nj, 3)   # 0..2
    out["criteria_mean_0_1"] = round(
        sum(out[c] for c in CRITERIA) / (len(CRITERIA) * 2), 4)
    out["acceptance_score"] = round(
        sum(VERDICT_VALUE[v["overall_verdict"]] for v in judged) / nj, 4)
    as_is = sum(v["overall_verdict"] == "ACCEPT_AS_IS" for v in judged)
    usable = sum(v["overall_verdict"] in
                 ("ACCEPT_AS_IS", "ACCEPT_WITH_MINOR_EDITS") for v in judged)
    rej = sum(v["overall_verdict"] == "REJECT" for v in judged)
    out["pct_accept_as_is"] = round(as_is / nj, 4)
    out["pct_usable"] = round(usable / nj, 4)
    out["pct_reject"] = round(rej / nj, 4)

    fails = {}
    for v in judged:
        if v["overall_verdict"] == "REJECT":
            t = v.get("primary_failure", "none")
            fails[t] = fails.get(t, 0) + 1
    out["top_failure"] = max(fails, key=fails.get) if fails else "none"
    return out

def show(r):
    print(f"\n=== {r['model']} ===")
    print(f"judged_ok={r['judged_ok']}/{r['total']}  judge_fail={r['judge_fail']}")
    if r["judged_ok"]:
        print("  per-criterion (0-2): " +
              "  ".join(f"{c}={r[c]}" for c in CRITERIA))
        print(f"  criteria_mean(0-1) = {r['criteria_mean_0_1']}")
        print(f"  acceptance_score   = {r['acceptance_score']}  "
              f"(AS_IS={r['pct_accept_as_is']:.0%}, "
              f"usable={r['pct_usable']:.0%}, reject={r['pct_reject']:.0%})")
        print(f"  top failure tag    = {r['top_failure']}")

RESULTS = {}

## 6–11. По одной ячейке на модель

Кэш на диске делает повторный запуск дешёвым; для смок-теста поставьте
`MAX_ITEMS = 20` в разделе 1.

In [7]:
RESULTS["qwen2.5-3b"] = evaluate_model("qwen2.5-3b")
show(RESULTS["qwen2.5-3b"])

qwen2.5-3b:   0%|          | 0/300 [00:00<?, ?it/s]

qwen2.5-3b: 100%|██████████| 300/300 [59:57<00:00, 11.99s/it]  


=== qwen2.5-3b ===
judged_ok=300/300  judge_fail=0
  per-criterion (0-2): C1=1.773  C2=1.61  C3=1.057  C4=1.033  C5=0.72  C6=0.837  C7=1.287
  criteria_mean(0-1) = 0.5941
  acceptance_score   = 0.155  (AS_IS=2%, usable=29%, reject=71%)
  top failure tag    = off_target


In [8]:
RESULTS["claude-opus-4-7"] = evaluate_model("claude-opus-4-7")
show(RESULTS["claude-opus-4-7"])

claude-opus-4-7: 100%|██████████| 300/300 [57:02<00:00, 11.41s/it] 


=== claude-opus-4-7 ===
judged_ok=300/300  judge_fail=0
  per-criterion (0-2): C1=1.967  C2=1.943  C3=1.84  C4=1.383  C5=1.237  C6=1.253  C7=1.82
  criteria_mean(0-1) = 0.8174
  acceptance_score   = 0.45  (AS_IS=20%, usable=70%, reject=30%)
  top failure tag    = off_target


In [9]:
RESULTS["smollm3-3b"] = evaluate_model("smollm3-3b")
show(RESULTS["smollm3-3b"])

smollm3-3b: 100%|██████████| 300/300 [1:06:15<00:00, 13.25s/it]


=== smollm3-3b ===
judged_ok=300/300  judge_fail=0
  per-criterion (0-2): C1=1.787  C2=1.603  C3=0.853  C4=0.993  C5=0.76  C6=0.79  C7=1.237
  criteria_mean(0-1) = 0.5731
  acceptance_score   = 0.1333  (AS_IS=4%, usable=23%, reject=77%)
  top failure tag    = off_target


In [10]:
RESULTS["phi3.5-mini"] = evaluate_model("phi3.5-mini")
show(RESULTS["phi3.5-mini"])

phi3.5-mini: 100%|██████████| 300/300 [53:48<00:00, 10.76s/it] 


=== phi3.5-mini ===
judged_ok=300/300  judge_fail=0
  per-criterion (0-2): C1=1.377  C2=0.677  C3=0.66  C4=0.697  C5=0.71  C6=0.403  C7=0.133
  criteria_mean(0-1) = 0.3326
  acceptance_score   = 0.0033  (AS_IS=0%, usable=1%, reject=99%)
  top failure tag    = malformed_structure


In [11]:
RESULTS["baseline_fib_v2"] = evaluate_model("baseline_fib_v2")
show(RESULTS["baseline_fib_v2"])

baseline_fib_v2: 100%|██████████| 300/300 [1:02:06<00:00, 12.42s/it]


=== baseline_fib_v2 ===
judged_ok=300/300  judge_fail=0
  per-criterion (0-2): C1=1.39  C2=1.223  C3=1.083  C4=0.38  C5=0.017  C6=0.54  C7=0.84
  criteria_mean(0-1) = 0.3909
  acceptance_score   = 0.0  (AS_IS=0%, usable=0%, reject=100%)
  top failure tag    = off_target


In [12]:
RESULTS["baseline_recon_v2"] = evaluate_model("baseline_recon_v2")
show(RESULTS["baseline_recon_v2"])

baseline_recon_v2: 100%|██████████| 300/300 [1:06:53<00:00, 13.38s/it]


=== baseline_recon_v2 ===
judged_ok=300/300  judge_fail=0
  per-criterion (0-2): C1=1.423  C2=1.147  C3=1.11  C4=2.0  C5=0.003  C6=0.153  C7=0.67
  criteria_mean(0-1) = 0.4647
  acceptance_score   = 0.0  (AS_IS=0%, usable=0%, reject=100%)
  top failure tag    = off_target


## 12. Сводная таблица

In [13]:
import pandas as pd

cols = ["model", "judged_ok", "judge_fail",
        "C1", "C2", "C3", "C4", "C5", "C6", "C7",
        "criteria_mean_0_1", "acceptance_score",
        "pct_accept_as_is", "pct_usable", "pct_reject", "top_failure"]
df = pd.DataFrame([RESULTS[m] for m in RESULTS], columns=cols)
df

,model,judged_ok,judge_fail,C1,C2,C3,C4,C5,C6,C7,criteria_mean_0_1,acceptance_score,pct_accept_as_is,pct_usable,pct_reject,top_failure
0,qwen2.5-3b,300,0,1.773,1.610,1.057,1.033,0.720,0.837,1.287,0.5941,0.1550,0.0233,0.2867,0.7133,off_target
1,claude-opus-4-7,300,0,1.967,1.943,1.840,1.383,1.237,1.253,1.820,0.8174,0.4500,0.2033,0.6967,0.3033,off_target
2,smollm3-3b,300,0,1.787,1.603,0.853,0.993,0.760,0.790,1.237,0.5731,0.1333,0.0367,0.2300,0.7700,off_target
3,phi3.5-mini,300,0,1.377,0.677,0.660,0.697,0.710,0.403,0.133,0.3326,0.0033,0.0000,0.0067,0.9933,malformed_structure
4,baseline_fib_v2,300,0,1.390,1.223,1.083,0.380,0.017,0.540,0.840,0.3909,0.0000,0.0000,0.0000,1.0000,off_target
5,baseline_recon_v2,300,0,1.423,1.147,1.110,2.000,0.003,0.153,0.670,0.4647,0.0000,0.0000,0.0000,1.0000,off_target


In [14]:
df.to_csv("judge_quality_results.csv", index=False, encoding="utf-8")
print("Сохранено: judge_quality_results.csv")

Сохранено: judge_quality_results.csv


## 13. Замечания

- 6 моделей × 300 = 1800 вызовов Kimi. Кэш в `judge_cache/` делает докатку
  дешёвой; стартуйте с `MAX_ITEMS = 20`.
- Покрытие не считается: шаблонные методы переписаны на 300/300, пробела
  охвата нет. Сравнение симметрично по всем 300 у всех шести моделей.
- Судья видит только `source` и само упражнение. `target`, `error_type`,
  `task_type` ему НЕ передаются — это чистый intrinsic по Amidei (2018) и
  защита от возврата `task_type_match` через чёрный ход.
- Числа судьи — индикатор, не доказательство, пока не откалиброваны против
  преподавателей (Cohen's kappa по `scores` и `overall_verdict`). Это
  следующий шаг для §2.1/§4.5.
- `claude-opus-4-7` здесь — оцениваемый генератор, не судья; судья всегда Kimi.